# grouping_V2: Alias-normalized grouping

Loads the original groups produced by `grouping.ipynb`, applies alias normalization
to the `Reduced` column (common-name → HD number, zero-padding fixes, HE tilde, etc.),
and re-assigns group IDs so previously split groups are merged.


In [24]:
import re
import pandas as pd
import numpy as np
from statistics import mode

# ── Alias normalization ───────────────────────────────────────────────────────
# Maps non-canonical reduced names → canonical form.
# Keys = aliases confirmed in the data; values = preferred name.
_ALIAS_MAP = {
    'betapic':      'hd39060',
    'betapictoris': 'hd39060',
    'spica':        'alphavir',
    'tetaql':       'thetaaql',
    '49cet':        '49ceti',
    'alphalyr':     'vega',
    'hd216956':     'fomalhaut',
    'betelgeuse':   'hd39801',
    'hd143275':     'deltasco',
}
_HD_ZERO  = re.compile(r'^(hd)0+(\d+)(.*)$')
_HIP_ZERO = re.compile(r'^(hip)0+(\d+)$')
_CS_ZERO  = re.compile(r'^(cs\d{5})0(\d{3,4})$')
_CD_D     = re.compile(r'^(cd-\d+)d(\d+)$')

def normalize_reduced(name):
    """Return the canonical reduced name, resolving known aliases and catalog formatting."""
    if not isinstance(name, str):
        return name
    if name in _ALIAS_MAP:
        return _ALIAS_MAP[name]
    m = _HD_ZERO.match(name)
    if m:
        return m.group(1) + m.group(2) + m.group(3)
    m = _HIP_ZERO.match(name)
    if m:
        return m.group(1) + m.group(2)
    if name.startswith('he~'):
        return 'he' + name[3:]
    m = _CS_ZERO.match(name)
    if m:
        return m.group(1) + m.group(2)
    m = _CD_D.match(name)
    if m:
        return m.group(1) + m.group(2)
    return name

# ── Load original outputs from grouping.ipynb ─────────────────────────────────
full_path   = '/home/msp25gd/Downloads/res/meta/full_metadata.pkl'
groups_path = '/home/msp25gd/Downloads/res/meta/groups.pkl'

full   = pd.read_pickle(full_path).copy()
groups = pd.read_pickle(groups_path).copy()

print(f'Loaded full_metadata: {len(full)} rows, {full["New Groups"].nunique()} groups')
print(f'Loaded groups:        {len(groups)} rows')

# ── Apply alias normalization to Reduced column ───────────────────────────────
before_unique = full['Reduced'].nunique()
full['Reduced'] = full['Reduced'].map(normalize_reduced)
after_unique = full['Reduced'].nunique()
print(f'\nAlias normalization: {before_unique} → {after_unique} unique names ({before_unique - after_unique} merged)')

# ── Re-assign group IDs based on normalized Reduced ──────────────────────────
# Build a mapping: normalized_name → new group ID
norm_names = full['Reduced'].unique()
name_to_id = {name: i for i, name in enumerate(sorted(norm_names))}
full['New Groups'] = full['Reduced'].map(name_to_id)

# Rebuild groups summary
full_grouped = full.groupby('New Groups')
all_groups = full_grouped.first()

if 'Sanitised' in full.columns:
    all_groups['Sanitised'] = full_grouped['Sanitised'].apply(lambda x: mode(x))
if 'Reduced' in full.columns:
    all_groups['Reduced'] = full_grouped['Reduced'].apply(lambda x: mode(x))
if 'Object' in full.columns:
    all_groups['Object'] = full_grouped['Object'].apply(lambda x: mode(x))

all_groups = all_groups.drop(
    columns=['SNR', 'Date', 'Checked', 'Coordtype', 'Epoch', 'Epochsystem', 'Equinox',
             'Parallax', 'PM Alpha', 'PM Delta', 'Airmass', 'RA', 'Dec'],
    errors='ignore'
)
all_groups = all_groups.reset_index(drop=False)

print(f'\nGroups after normalization: {full["New Groups"].nunique()}')
sizes = full.groupby('New Groups').size()
print(f'Groups with < 3 rows: {(sizes < 3).sum()}')
print(f'Groups with < 2 rows: {(sizes < 2).sum()}')

# ── Save outputs ──────────────────────────────────────────────────────────────
out_full_path   = '/home/msp25gd/Downloads/res/meta/full_metadata_V2.pkl'
out_groups_path = '/home/msp25gd/Downloads/res/meta/groups_V2.pkl'

full.to_pickle(out_full_path)
all_groups.to_pickle(out_groups_path)
print(f'\nSaved: {out_full_path}')
print(f'Saved: {out_groups_path}')


Loaded full_metadata: 32308 rows, 12827 groups
Loaded groups:        12827 rows

Alias normalization: 12827 → 12799 unique names (28 merged)

Groups after normalization: 12799
Groups with < 3 rows: 9915
Groups with < 2 rows: 7197

Saved: /home/msp25gd/Downloads/res/meta/full_metadata_V2.pkl
Saved: /home/msp25gd/Downloads/res/meta/groups_V2.pkl
